# A LangGraph agent end to end

Booster LEAP, Agentic AI, topic 6. Run a graph with a state, a checked router, a retried lookup, an approval pause with `interrupt`, an idempotent refund, a checkpointer, a resume and a step limit, read the trace of every update, change the program, and (optionally) give the router a real model with your own key.

<h1 style="color:#b45309">&#9888; First: File &rarr; Save a copy in Drive</h1>

<span style="font-size:20px">This notebook is read-only. Make your own copy now, then work in the copy (no GitHub login needed).</span>

**How to use this notebook:** run the cells from top to bottom (Runtime, Run all, or Shift+Enter on each cell). Change one thing at a time and run again; the experiments tell you what to try.

## Stage 1: the program with a stand-in router model

The only **stand-in** in stage 1 is the model that picks the route: a fake chat model from `langchain_core` that returns a prepared word. The graph, the state with its reducer, the retry policy, the interrupt, the checkpointer and the streaming are the real `langgraph` package. The order service, the policy and the payment ledger are small stand-ins for real systems. Run the next cell once; it installs the packages.

In [ ]:
# Installs LangGraph and LangChain (the outputs below were produced with langgraph 1.2.24, langchain 1.5.2 and langchain-core 1.8.4;
# a later release can differ in small details). Colab may ask you to restart the runtime after an install: if so, do it and run the cells again.
%pip install -q langgraph langchain

The setup cell: imports, the support desk data (orders with owners, one policy passage), the state (a `TypedDict` whose `notes` list uses a reducer, so every node *adds* a line), a checkpoint store (`InMemorySaver`), a ledger of issued refunds, a switch that makes the order service fail once, and the stand-in model. It prints nothing.

In [ ]:
import operator
from typing import Annotated, TypedDict

from langchain_core.language_models.fake_chat_models import GenericFakeChatModel
from langchain_core.messages import AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, RetryPolicy, interrupt

CUSTOMER = "C2"                       # the customer who is logged in (topic 2: identity comes from the session, not from the model)
ORDERS = {"A1001": {"owner": "C1", "status": "shipped"}, "A1002": {"owner": "C2", "status": "delivered"}, "A1003": {"owner": "C1", "status": "processing"}}
POLICY = "Refunds are allowed within 30 days of delivery, for orders in the status delivered [p1]."
ROUTES = ["status", "refund"]
SAVER = InMemorySaver()               # the checkpoint store: it lives in this process only (a real service needs a durable one)
LEDGER = {}                           # refunds that were really issued: idempotency key -> order id (a stand-in for the payment system)
FLAKY = {"failures_left": 1}          # the order service fails once with a connection error, to show the retry policy

class State(TypedDict):
    request: str
    order_id: str
    route: str
    status: str
    approved: bool
    notes: Annotated[list, operator.add]       # reducer: every node ADDS its note
    answer: str

class StandIn(GenericFakeChatModel):
    """A stand-in for the MODEL only: it returns the prepared replies in order and ignores its input."""

def model(*replies):
    return StandIn(messages=iter([AIMessage(content=r) for r in replies]))

The graph. `build` defines six nodes and their edges and compiles them with the checkpointer. `router` asks the model for one word and checks it against a closed list. `lookup` applies the ownership rule and carries a `RetryPolicy`. `approval` calls `interrupt`, which pauses the run and hands a request to a person. `refund` uses an idempotency key. Two edges are conditional: after `lookup` and after `approval`. It prints nothing.

In [ ]:
def build(router_model):
    def router(state: State):
        """The model picks a route; the code checks it against the closed list (lesson 5)."""
        choice = router_model.invoke([("system", "Reply with one word: status or refund."), ("human", state["request"])]).content.strip()
        return {"route": choice if choice in ROUTES else "status", "notes": [f"router: {choice}"]}

    def lookup(state: State):
        """The tool node. The ownership rule is ours, not the framework's (topic 2). The service fails once; the retry policy tries again."""
        if FLAKY["failures_left"] > 0:
            FLAKY["failures_left"] -= 1
            print("   (lookup: the order service is busy, the retry policy will call the node again)")
            raise ConnectionError("order service busy")
        order = ORDERS.get(state["order_id"])
        if order is None or order["owner"] != CUSTOMER:
            return {"status": "unknown", "notes": [f"lookup: no order {state['order_id']} on this account"]}
        return {"status": order["status"], "notes": [f"lookup: {state['order_id']} is {order['status']}"]}

    def policy(state: State):
        return {"notes": ["policy: " + POLICY]}

    def approval(state: State):
        """Pauses the run. The value goes to the person; what comes back from Command(resume=...) is the return value of interrupt()."""
        decision = interrupt({"action": "issue_refund", "order_id": state["order_id"], "status": state["status"], "rule": POLICY})
        return {"approved": bool(decision.get("approved")), "notes": [f"approval: {'approved' if decision.get('approved') else 'declined'}"]}

    def refund(state: State):
        key = f"refund-{state['order_id']}"
        if key in LEDGER:                                       # idempotent: the same key never pays twice
            return {"notes": [f"refund: {key} was already issued, nothing sent"]}
        LEDGER[key] = state["order_id"]
        return {"notes": [f"refund: {key} issued"]}

    def reply(state: State):
        return {"answer": state["notes"][-1]}

    builder = StateGraph(State)
    builder.add_node("router", router)
    builder.add_node("lookup", lookup, retry_policy=RetryPolicy(max_attempts=3, initial_interval=0.01, jitter=False, retry_on=ConnectionError))
    builder.add_node("policy", policy)
    builder.add_node("approval", approval)
    builder.add_node("refund", refund)
    builder.add_node("reply", reply)
    builder.add_edge(START, "router")
    builder.add_edge("router", "lookup")
    builder.add_conditional_edges("lookup", lambda s: "policy" if s["route"] == "refund" and s["status"] == "delivered" else "reply", {"policy": "policy", "reply": "reply"})
    builder.add_edge("policy", "approval")
    builder.add_conditional_edges("approval", lambda s: "refund" if s["approved"] else "reply", {"refund": "refund", "reply": "reply"})
    builder.add_edge("refund", "reply")
    builder.add_edge("reply", END)
    return builder.compile(checkpointer=SAVER)

Six runs. `start` and `follow` stream the updates of a run and print one line per node (the trace), and print the pause when there is one. Run 3 pauses for approval and is resumed with `Command(resume=...)` on a new graph object that shares only the saver and the thread id. Run 4 repeats the refund on a new thread and shows that the payment key stops a second refund. Run 5 sets a step limit that is too low.

In [ ]:
def start(graph, thread, request, order_id, **config):
    """Run until the graph ends or pauses, and print every update of the stream (the trace)."""
    cfg = {"configurable": {"thread_id": thread}, **config}
    first = {"request": request, "order_id": order_id, "notes": [], "approved": False}
    return follow(graph, cfg, first)

def follow(graph, cfg, first):
    for update in graph.stream(first, cfg, stream_mode="updates"):
        for node, change in update.items():
            if node == "__interrupt__":
                print("   PAUSED, waiting for a person:", change[0].value)
            else:
                print(f"   {node:8} -> {sorted(change)}")
    state = graph.get_state(cfg)
    print("   next:", state.next or "(finished)", "| answer:", state.values.get("answer", "-"))
    return cfg

print("1. a status question; the order service fails once and the retry policy tries again")
start(build(model("status")), "t-1", "Where is my order A1002?", "A1002")

print("2. an order that belongs to another customer")
start(build(model("status")), "t-2", "Where is my order A1003?", "A1003")

print("3. a refund: the run pauses for approval, then a person resumes it")
graph = build(model("refund"))
cfg = start(graph, "t-3", "Refund A1002, it arrived broken.", "A1002")
print("   (a new graph object is built here; only the saver and the thread id connect it to the paused run)")
graph = build(model())
follow(graph, cfg, Command(resume={"approved": True}))
print("   refunds issued so far:", LEDGER)

print("4. the same refund again on a new thread: the key stops a second payment")
graph = build(model("refund"))
cfg = start(graph, "t-4", "Refund A1002, it arrived broken.", "A1002")
follow(graph, cfg, Command(resume={"approved": True}))
print("   refunds issued so far:", LEDGER)

print("5. a step limit: the same status question with recursion_limit=2")
try:
    start(build(model("status")), "t-5", "Where is my order A1002?", "A1002", recursion_limit=2)
except GraphRecursionError as error:
    print("   stopped:", type(error).__name__)

print("6. what the checkpointer kept for thread t-3:", len(list(graph.get_state_history({"configurable": {"thread_id": "t-3"}}))), "snapshots")

1. a status question; the order service fails once and the retry policy tries again
   router   -> ['notes', 'route']
   (lookup: the order service is busy, the retry policy will call the node again)
   lookup   -> ['notes', 'status']
   reply    -> ['answer']
   next: (finished) | answer: lookup: A1002 is delivered
2. an order that belongs to another customer
   router   -> ['notes', 'route']
   lookup   -> ['notes', 'status']
   reply    -> ['answer']
   next: (finished) | answer: lookup: no order A1003 on this account
3. a refund: the run pauses for approval, then a person resumes it
   router   -> ['notes', 'route']
   lookup   -> ['notes', 'status']
   policy   -> ['notes']
   PAUSED, waiting for a person: {'action': 'issue_refund', 'order_id': 'A1002', 'status': 'delivered', 'rule': 'Refunds are allowed within 30 days of delivery, for orders in the status delivered [p1].'}
   next: ('approval',) | answer: -
   (a new graph object is built here; only the saver and the thread id co

## Experiments (stage 1)

Change one thing at a time, predict the output, then run again (run the setup cell first when you change it, and the graph cell when you change a node).

1. Resume run 3 with `{"approved": False}`. What does the reply say, and what is in the ledger?
2. Set `FLAKY = {"failures_left": 2}` in the setup cell, then `3`. What happens in each case, and why?
3. Change the router's prepared word in run 1 to `billing`. Which route does the program take, and why?
4. Run the status question with `recursion_limit` 2, 3 and 4. Which limits stop it?
5. Set `CUSTOMER = "C1"` and run the refund request for A1002.
6. In the graph cell, compile without the checkpointer (`builder.compile()`) and run the refund request. What happens at the approval, and what happens when you resume with `Command(resume=...)`?

## Stage 2 (optional): a real model in the router

Stage 1 used a stand-in router. This stage gives the same graph a real model with **your own key**. Read before you run:

- It sends the request text to the provider and costs a small amount of money. Each run is one short call, but you pay for what you run.
- Never write the key into a cell or share a notebook that contains it. The cell reads it from Colab's Secrets.
- A real model may reply with a sentence instead of one word. The router checks the reply against the list and falls back to `status`, so look at the `router:` note in the trace to see what the model really said.
- The nodes, the ownership rule, the checkpointer, the payment key and the step limit are the same code.

In [ ]:
# Optional stage 2: the stand-in model of the router is replaced by a real model. This sends the request text to the provider and costs a little money.
# One-time set-up: open the key symbol (Secrets) in the left bar of Colab, add a secret named OPENAI_API_KEY and switch "Notebook access" on.
%pip install -q "langchain[openai]"
import os
from google.colab import userdata
from langchain.chat_models import init_chat_model

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
MODEL = "PUT-A-MODEL-NAME-HERE"       # a chat model of your account (see the provider's documentation)
assert MODEL != "PUT-A-MODEL-NAME-HERE", "choose a model first"
real_router = init_chat_model(MODEL, model_provider="openai")

In [ ]:
LEDGER.clear()
FLAKY["failures_left"] = 0
graph = build(real_router)                                # only the router's model changed

print("a status question")
start(graph, "real-1", "Where is my order A1002?", "A1002")

print("a refund request")
cfg = start(graph, "real-2", "I want my money back for A1002, it arrived broken.", "A1002")
if graph.get_state(cfg).next:                             # the run paused at the approval
    follow(graph, cfg, Command(resume={"approved": True}))
print("refunds issued:", LEDGER)

## What to look at

1. What did the real model say in the `router:` note for each request? Was it one of the two words, or did the check have to fall back?
2. Did the refund request reach the approval pause? What did the request to the person contain?
3. Run the refund request a second time on a new thread id. Did the ledger change?
4. Write down one case where the real model and the stand-in differ, and which part of the program protected you.